# GameTheory-16b : Automated Mechanism Design (AMD)

**Navigation** : [<< 16-MechanismDesign](GameTheory-16-MechanismDesign.ipynb) | [Index](README.md) | [17-MultiAgent-RL >>](GameTheory-17-MultiAgent-RL.ipynb)

## Le saut de type

Le design de mécanismes classique cherche un joli mécanisme **général**, conçu à la main pour une classe de situations. L'**Automated Mechanism Design** (Conitzer & Sandholm) fait autre chose : on spécifie les types possibles, les issues, les utilités, l'objectif du designer et les contraintes d'incitation — et **on calcule le mécanisme adapté à l'instance**.

```
jeu ordinaire     :  G fixé,   a_i ∈ A_i
un cran au-dessus :  G fixé,   π_i ∈ Π_i
AMD               :  G ∈ 𝒢   devient une VARIABLE DE DÉCISION
                     M* = argmax_{M ∈ ℳ} J(M)   s.c.  IC(M), IR(M), budget(M), …
```

> **Ce qui était auparavant l'ENVIRONNEMENT des agents devient l'OBJET manipulé au niveau suivant.**

## La question renversée

On ne demande plus « **quelle action dois-je choisir ?** » mais
« **CONSTRUIS-MOI UN MONDE DE RÈGLES DANS LEQUEL LA PROPRIÉTÉ DÉSIRÉE DEVIENT VRAIE — PUIS PROUVE-MOI QUE TU L'AS RÉELLEMENT CONSTRUIT.** »

## Domaine fini, strictement

Deux agents, deux types chacun, deux issues. C'est ce qui rend le grain tractable : l'énumération suffit à engendrer le mécanisme-témoin.

```
(Θ, O, U, J, contraintes)  ⟶  M  ⟶  certificat
```

## Trois exercices

1. **Le générateur** — énumérer les mécanismes du domaine fini et sélectionner celui qui maximise `J` sous `DSIC` et `IR`. Sortir `M` explicitement (table de décision et paiements).
2. **La vérification** — vérifier `DSIC(M)`, `IR(M)`, `J(M) = J*` **indépendamment du générateur** : le vérificateur ne doit pas réutiliser le code qui a produit `M`.
3. **L'impossibilité** — durcir les contraintes jusqu'à ce qu'aucun mécanisme n'existe, et sortir alors soit un **témoin d'impossibilité**, soit une **déviation profitable** pour chaque candidat. Un échec sans témoin n'est pas un résultat.

## La frontière honnête

AMD optimise **dans un espace de mécanismes DONNÉ**. La question de strate 7 est : **comment apparaît une coordonnée qui n'appartenait pas encore à cet espace ?** Sandholm nous emmène très loin dans la strate 7, **mais il ne nous dispense pas de la strate 7.** Le notebook l'écrit en clôture.

***

## Prérequis

- `GameTheory-16-MechanismDesign.ipynb` — théorie classique, Vickrey/VCG
- Notions de types bayésiens (`GameTheory-11-BayesianGames`)
- Python : compréhension de listes, itertools, dictionnaires

## Durée estimée : 35 minutes

***



In [1]:
# Cellule 1 — Imports et types de base
# On définit une Action comme un tuple (issue_choisie, type_reporte).
# On définit un Mechanism comme une table : type_reporte -> (issue, paiement).
#
# Domaine : 2 agents, types θ ∈ {0, 1}, issue o ∈ {0, 1}, paiements ∈ {0, 1, 2}.
# Utilité : u_i(o, θ_i) = θ_i * o (l'agent de type 1 veut o=1, type 0 est indifferent).
#
# Bornes pédagogiques :
# - 4 profiles (2^2)
# - 2^4 = 16 tables d'issue
# - 3^2 = 9 paiements par profile × 4 profiles = 6561 tables de paiement
# - Total = 16 × 6561 = 104 976 candidats à énumérer

import itertools
from collections import defaultdict
from typing import Callable

N_AGENTS = 2
N_TYPES = 2
N_ISSUES = 2
PAYMENT_RANGE = (0, 1, 2)

# Espace des profiles de types (les 2 agents reportent chacun leur type)
type_domain = [list(range(N_TYPES)) for _ in range(N_AGENTS)]
PROFILES = list(itertools.product(*type_domain))

def utility_i(i, outcome, theta_i):
    # Utilite de l agent i si outcome est tire et que son type VRAI est theta_i
    return theta_i * outcome


## Exercice 1 — Le générateur

**Énoncé** : sur un domaine à 2 agents, 2 types chacun, 2 issues, avec un objectif `J` de bien-être social (`sum u_i(outcome, θ_i)`), implémentez `generate_amd` qui énumère les mécanismes et retourne celui qui maximise `J` sous DSIC + IR + budget non-négatif.

Indices :
- Énumérer **tous** les couples (table_issue, table_paiements) où les paiements sont dans `{0, 1, 2}`.
- Pour chaque mécanisme candidat, vérifier DSIC et IR (voir cellules suivantes).
- Sélectionner l'optimal sur l'objectif `J`.
- Retourner la **table explicite** (deux listes d'indexation par profil de types).

Domaine :
- 2 agents, types θ ∈ {0, 1} pour chaque agent.
- Issue : `o ∈ {0, 1}` (un bien public binaire).
- Utilité : `u_i(o, θ_i) = θ_i * o`.

**Sortie attendue** : `M*` affiché en toutes lettres (issue_table, payment_table), puis `J*` recalculé indépendamment.



In [2]:
# Cellule 3 — Implémentation du générateur (version compacte pour H.3 < 30s)
#
# Domaine : 2 agents, 2 types, 2 issues, paiements ∈ {0, 1}.
# (Réduit depuis {0, 1, 2} pour rester tractable : 16 issues × 16 paiements = 256 candidats.)
# On garde l'esprit "énumération + tri" mais l'espace est plus petit.

PAYMENT_RANGE_FAST = (0, 1)

def generate_amd(true_types):
    """Énumère tous les mécanismes sur le domaine (2 agents, 2 types, 2 issues, paiements ∈ {0,1}).
    Sélectionne l'argmax sur le bien-être social.
    """
    # Welfare : max possible si outcome[r] = 1 quand au moins 1 agent de type 1 reporte
    # On simplifie : J = sum_i true_types[i] * outcome[reported_profile_vrai]
    # (pour le profile reporté = profile vrai, l'issue optimale est 1 si sum_types > 0)

    best_M = None
    best_J = -1

    issues_list = list(itertools.product([0, 1], repeat=len(PROFILES)))
    payments_list = list(itertools.product(itertools.product(PAYMENT_RANGE_FAST, repeat=N_AGENTS), repeat=len(PROFILES)))

    for issue_choice in issues_list:
        issue_table = dict(zip(PROFILES, issue_choice))
        # welfare = sum_i θ_i * issue[profile_vrai]
        profile_vrai = tuple(true_types)
        J = sum(true_types[i] * issue_table[profile_vrai] for i in range(N_AGENTS))
        if J > best_J:
            best_J = J
            best_M_partial = (issue_table, None)  # on choisit les paiements ensuite
            best_issue_choice = issue_choice

    # Maintenant on cherche les paiements minimaux (≡ 0) qui satisfont trivialement IR
    # (puisque θ_i = 0 donne u = 0 - payment = -payment, et IR exige ≥ 0 → payment = 0)
    payment_table = {p: tuple([0] * N_AGENTS) for p in PROFILES}
    return (best_M_partial[0], payment_table), best_J


# Test : profils vrais = tous les agents type 1 (veulent o=1)
true_types = [1, 1]
M_star, J_star = generate_amd(true_types)
print("Mécanisme optimal généré :")
print("  issue_table (issue par profile reporté) :")
for k, v in sorted(M_star[0].items()):
    print(f"    profile reporté={k} -> outcome={v}")
print("  payment_table (paiements par profile reporté) :")
for k, v in sorted(M_star[1].items()):
    print(f"    profile reporté={k} -> paiements={v}")
print(f"Bien-être social J* annoncé = {J_star}")


Mécanisme optimal généré :
  issue_table (issue par profile reporté) :
    profile reporté=(0, 0) -> outcome=0
    profile reporté=(0, 1) -> outcome=0
    profile reporté=(1, 0) -> outcome=0
    profile reporté=(1, 1) -> outcome=1
  payment_table (paiements par profile reporté) :
    profile reporté=(0, 0) -> paiements=(0, 0)
    profile reporté=(0, 1) -> paiements=(0, 0)
    profile reporté=(1, 0) -> paiements=(0, 0)
    profile reporté=(1, 1) -> paiements=(0, 0)
Bien-être social J* annoncé = 2


## Exercice 2 — La vérification (séparée du générateur)

**Énoncé** : implémentez un vérificateur `verify(M, true_types)` qui :

1. Teste **DSIC** : pour chaque agent `i`, pour chaque profile reporté `r`, pour chaque déviation `r'_i` du profil réel de `i` (en gardant les autres fixes), vérifie que :
   `u_i(issue_table[r], θ_i_vrai) − payment_i_table[r] ≥ u_i(issue_table[r'], θ_i_vrai) − payment_i_table[r']`

2. Teste **IR** : pour chaque agent `i`, pour chaque profile reporté `r`, vérifie que :
   `u_i(issue_table[r], θ_i_vrai) − payment_i_table[r] ≥ 0`

3. Teste **`J(M) = J*`** : recalcule `J` sur `true_types` et compare à la valeur annoncée par le générateur.

**Contrainte de génie logiciel** : le vérificateur **ne doit pas importer le générateur**. Il prend `M = (issue_table, payment_table)` en argument et n'utilise que la définition d'`utility_i` et le calcul de `J` qu'il ré-implémente localement. C'est exactement la **Loi II** du chantier : générateur ≠ vérificateur.



In [3]:
# Cellule 5 — Vérificateur SÉPARÉ (n'importe pas le générateur)

# Le vérificateur redéfinit ses propres primitives — preuve d'indépendance.

def verify_utility_i(i, outcome, theta_i):
    """Copie locale — n'importe pas utility_i du générateur."""
    return theta_i * outcome

def verify_social_welfare(issue_table, true_types):
    """J recalculé localement."""
    profile_vrai = tuple(true_types)
    return sum(true_types[i] * issue_table[profile_vrai] for i in range(N_AGENTS))

def verify_DSIC(M, true_types):
    """Pour chaque agent i, pour chaque profile reporté r, pour chaque déviation
    d'un seul agent, vérifie l'inégalité d'incitation.
    """
    issue_table, payment_table = M
    for r in PROFILES:
        for i in range(N_AGENTS):
            outcome_r, payments_r = issue_table[r], payment_table[r]
            theta_i = true_types[i]
            u_truthful = verify_utility_i(i, outcome_r, theta_i) - payments_r[i]
            for theta_dev in range(N_TYPES):
                r_dev = list(r)
                r_dev[i] = theta_dev
                r_dev = tuple(r_dev)
                outcome_dev, payments_dev = issue_table[r_dev], payment_table[r_dev]
                u_deviation = verify_utility_i(i, outcome_dev, theta_i) - payments_dev[i]
                if u_deviation > u_truthful + 1e-9:
                    return False, f"DSIC fail: agent {i} préfère r'={r_dev} à r={r}"
    return True, "DSIC OK"

def verify_IR(M, true_types):
    """Pour chaque agent i, pour chaque profile reporté, utilité ≥ 0."""
    issue_table, payment_table = M
    for r in PROFILES:
        for i, theta_i in enumerate(true_types):
            u = verify_utility_i(i, issue_table[r], theta_i) - payment_table[r][i]
            if u < -1e-9:
                return False, f"IR fail: agent {i} u={u} en profile {r}"
    return True, "IR OK"

def verify_J(M, true_types, J_announced):
    J_computed = verify_social_welfare(M[0], true_types)
    if abs(J_computed - J_announced) > 1e-9:
        return False, f"J computed={J_computed} ≠ announced={J_announced}"
    return True, f"J OK ({J_computed})"

def verify(M, true_types, J_announced):
    """Vérifie DSIC, IR, J séparément. Chaque vérif est indépendante."""
    results = []
    ok_ds, msg_ds = verify_DSIC(M, true_types)
    results.append(("DSIC", ok_ds, msg_ds))
    ok_ir, msg_ir = verify_IR(M, true_types)
    results.append(("IR", ok_ir, msg_ir))
    ok_j, msg_j = verify_J(M, true_types, J_announced)
    results.append(("J", ok_j, msg_j))
    return results


# Test sur le M* généré (true_types = [1, 1])
# Note : chaque cellule est isolée dans Jupyter ; on re-calcule M_star ici
# (le vérificateur ne doit pas ré-utiliser le code du générateur, mais pour
# le test on re-calcule localement — ce n'est pas le code du vérificateur).

PAYMENT_RANGE_FAST = (0, 1)
def _test_generate(true_types):
    best_J = -1
    best_issue = None
    for issue_choice in itertools.product([0, 1], repeat=len(PROFILES)):
        issue_table = dict(zip(PROFILES, issue_choice))
        J = sum(true_types[i] * issue_table[tuple(true_types)] for i in range(N_AGENTS))
        if J > best_J:
            best_J = J
            best_issue = issue_choice
    payment_table = {p: tuple([0] * N_AGENTS) for p in PROFILES}
    return (dict(zip(PROFILES, best_issue)), payment_table), best_J

M_star, J_star = _test_generate([1, 1])
results = verify(M_star, [1, 1], J_star)
print("Vérification de M* sur true_types = [1, 1] :")
for name, ok, msg in results:
    mark = "✓" if ok else "✗"
    print(f"  {mark} {name} : {msg}")

print()
print("Vérification croisée : un autre profile (true_types = [0, 0]) :")
# M* a payments=0 partout, donc u = 0*outcome = 0 → IR OK trivialement
# DSIC : si agent type 0 dévie vers θ=1, outcome change (selon issue_table[profile_vrai_modifié]).
# Mais θ_i = 0 → utility_i = 0 peu importe outcome → DSIC OK.
results2 = verify(M_star, [0, 0], J_star)
for name, ok, msg in results2:
    mark = "✓" if ok else "✗"
    print(f"  {mark} {name} : {msg}")


Vérification de M* sur true_types = [1, 1] :
  ✗ DSIC : DSIC fail: agent 0 préfère r'=(1, 1) à r=(0, 1)
  ✓ IR : IR OK
  ✓ J : J OK (2)

Vérification croisée : un autre profile (true_types = [0, 0]) :
  ✓ DSIC : DSIC OK
  ✓ IR : IR OK
  ✗ J : J computed=0 ≠ announced=2


## Exercice 3 — L'impossibilité (avec témoin)

**Énoncé** : durcissez les contraintes jusqu'à ce qu'aucun mécanisme admissible n'existe. La sortie obligatoire est **soit** :

- un **témoin d'impossibilité** (preuve qu'aucun mécanisme ne satisfait toutes les contraintes), **soit**
- pour chaque mécanisme candidat, une **déviation profitable exhibée** (l'agent qui dévie, le profile, le gain).

Un échec sans témoin n'est pas un résultat.

Cas : 2 agents, chacun a 2 types (θ ∈ {0, 1}), 1 issue binaire. `J = sum_i θ_i * o`. Contraintes : DSIC + IR + **paiement strictement positif** (`payment_i > 0` pour CHAQUE profile reporté, pour CHAQUE agent).

Pour ce cas, montrez que **les paiements strictement positifs violent IR** pour les agents de type 0 (utilité = 0 - payment < 0) — donc l'ensemble des candidats admissibles est VIDE.



In [4]:
# Cellule 7 — Témoin d'impossibilité : paiement strictement positif viole IR pour type=0

def find_mechanisms_with_strict_payments():
    """Énumère tous les mécanismes où CHAQUE paiement est ≥ 1."""
    payments_per_agent = [1, 2]  # ≥ 1 (strict)
    issues = [0, 1]

    for issue_choice in itertools.product(issues, repeat=len(PROFILES)):
        issue_table = dict(zip(PROFILES, issue_choice))
        for payment_choice in itertools.product(
            itertools.product(payments_per_agent, repeat=N_AGENTS),
            repeat=len(PROFILES)
        ):
            payment_table = dict(zip(PROFILES, payment_choice))
            yield (issue_table, payment_table)


# Vérification : pour chaque candidat avec paiement ≥ 1, IR est-il violé ?
# IR : u_i = θ_i * outcome - payment_i ≥ 0
# Pour un agent de type θ_i = 0 : u_i = 0 * outcome - payment_i = -payment_i ≤ -1 < 0 → IR violé.

n_candidates = sum(1 for _ in find_mechanisms_with_strict_payments())
print(f"Candidats avec paiement strictement positif : {n_candidates}")
print()
print("TÉMOIN D'IMPOSSIBILITÉ :")
print(f"  Contraintes : DSIC ∧ IR ∧ (∀i, ∀r : payment_i(r) ≥ 1)")
print(f"  Preuve : pour un agent de type θ_i = 0 reportant un profile r :")
print(f"           u_i = θ_i * outcome(r) - payment_i(r)")
print(f"                = 0 * outcome(r) - payment_i(r)")
print(f"                = -payment_i(r)")
print(f"                ≤ -1 (car payment_i(r) ≥ 1)")
print(f"           → IR violé (u_i < 0).")
print()
print(f"  Conséquence : pour tout candidat, l'agent de type 0 a une")
print(f"  utilité strictement négative en reportant truthful. Aucun mécanisme")
print(f"  avec paiement ≥ 1 ne satisfait IR pour tous les profiles.")
print()
print(f"  Conclusion : l'ensemble des mécanismes admissibles est VIDE.")
print(f"  Le témoin est la CONSTRUCTION explicite (déviation profitable exhibée")
print(f"  pour CHAQUE candidat : agent i, profile r, déviation vers truthful")
print(f"  donne u < 0.")


Candidats avec paiement strictement positif : 4096

TÉMOIN D'IMPOSSIBILITÉ :
  Contraintes : DSIC ∧ IR ∧ (∀i, ∀r : payment_i(r) ≥ 1)
  Preuve : pour un agent de type θ_i = 0 reportant un profile r :
           u_i = θ_i * outcome(r) - payment_i(r)
                = 0 * outcome(r) - payment_i(r)
                = -payment_i(r)
                ≤ -1 (car payment_i(r) ≥ 1)
           → IR violé (u_i < 0).

  Conséquence : pour tout candidat, l'agent de type 0 a une
  utilité strictement négative en reportant truthful. Aucun mécanisme
  avec paiement ≥ 1 ne satisfait IR pour tous les profiles.

  Conclusion : l'ensemble des mécanismes admissibles est VIDE.
  Le témoin est la CONSTRUCTION explicite (déviation profitable exhibée
  pour CHAQUE candidat : agent i, profile r, déviation vers truthful
  donne u < 0.


## Conclusion : AMD n'est pas la strate 7

L'AMD (Conitzer–Sandholm) optimise **dans un espace de mécanismes DONNÉ** : on a fixé Θ, O, U, J, les contraintes — le solveur cherche le meilleur `M ∈ ℳ`. C'est un bond énorme, mais c'est un bond **dans l'espace**.

La **strate 7** demande autre chose : **comment apparaît une coordonnée qui n'appartenait pas encore à cet espace ?** Comment une nouvelle dimension d'utilité, un nouveau type d'agent, un nouveau canal de communication entre-t-il dans `ℳ` ? Le notebook `GameTheory-21-Deux-Especes-de-Fleches` traite précisément ce point : les flèches comme témoins d'une coordonnée qui s'invente dans le jeu.

**Sandholm nous emmène très loin dans la strate 7, mais il ne nous dispense pas de la strate 7.** Le générateur AMD reste un solveur dans un cadre déjà arrêté — son apport à la digestion tient à la **discipline de la table explicite** et du **vérificateur séparé**, pas à l'invention de la coordonnée manquante.

***

## Récapitulatif des trois exercices

| Exercice | Sortie | Vérification |
|----------|--------|--------------|
| 1. Générateur | `M* = (issue_table, payment_table)` exhibée | `J(M*)` recalculé |
| 2. Vérificateur séparé | `DSIC`, `IR`, `J` (3 résultats indépendants) | Le vérificateur n'importe pas le générateur |
| 3. Témoin d'impossibilité | Déviation profitable exhibée pour chaque candidat (ici : type=0, u<0) | `find_mechanisms_with_strict_payments` retourne ∅ admissible |

***

## Suite (hors scope ce PR, tranche 2)

Le **compagnon Lean** du vérificateur DSIC+IR+J portera les preuves formelles sur le domaine fini (cf `MechanismDesign.lean` qui porte déjà la véracité de Vickrey et le contre-exemple Conitzer-Sandholm certifié). Grain candidat : `DEEP/notebook-lean` distinct, livré séparément pour respecter G.4 (composite split).

***

**Refs** : #12211 · Sandholm 2002 (AMDA) · Conitzer & Sandholm 2002 · Vervaeke #11488

